# 04 - Preparação da base do soft sensor

**Objetivo:** Criar atributos temporais, defasagens e partições cronológicas para regressão.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento e seleção saudável

In [ ]:
d=carregar_csv(ROOT/'dados/02_intermediarios/dados_com_flags_qualidade.csv'); d=atributos_temporais(d); d=d[d.apto_treino_soft_sensor.eq(1)].copy(); print('Registros saudáveis:',len(d))

## Defasagens e médias passadas

In [ ]:
for lag in [1,3,6,12,24]:
    for c in ['temperatura_ambiente_C','radiacao_solar_global_W_m2','umidade_relativa_ar_pct']:
        d[f'{c}_lag_{lag}']=d[c].shift(lag)
for w in [6,12,24]:
    for c in ['temperatura_ambiente_C','radiacao_solar_global_W_m2','velocidade_vento_m_s']:
        d[f'{c}_media_{w}']=d[c].rolling(w,min_periods=w).mean()
d=d.dropna(subset=['temperatura_interna_C']).copy()

## Partições temporais

In [ ]:
q1=d.timestamp.quantile(.60); q2=d.timestamp.quantile(.80); d['particao']=np.where(d.timestamp<=q1,'treino',np.where(d.timestamp<=q2,'validacao','teste'))
features=['temperatura_ambiente_C','umidade_relativa_ar_pct','umidade_relativa_interna_pct','radiacao_solar_global_W_m2','temperatura_ponto_orvalho_C','velocidade_vento_m_s','chuva_acumulada_dia_mm','hora_seno','hora_cosseno','vento_dir_seno','vento_dir_cosseno']+[c for c in d.columns if '_lag_' in c or '_media_' in c]
d=d.dropna(subset=features).copy(); display(d.particao.value_counts().to_frame('registros')); print('Features:',len(features))

## Exportação

In [ ]:
cols=['timestamp','temperatura_interna_C','evento_referencia','particao']+features
base=d[cols]
out=ROOT/'dados/03_processados/base_soft_sensor.csv'
base.to_csv(out,index=False,encoding='utf-8-sig')
(ROOT/'resultados/tabelas/features_soft_sensor.txt').write_text(chr(10).join(features),encoding='utf-8')
print('Salvo:',out,base.shape)


## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.